# Fine-Tuning  
Este cuaderno entrena el modelo Qwen2.5-VL-7B a partir del dataset generado previamene. Antes de ejecutar este cuaderno, es necesario generar el dataset de entrenamiento, o copiar el proporcionado en Google Drive. Sigue los pasos descritos en preparar_dataset.txt en la carpeta api/Dataset/.

## 1. Instalar dependencias


In [ ]:

print('=' * 60)
print('=' * 60)
!pip install -q unsloth unsloth_zoo trl peft accelerate bitsandbytes xformers qwen-vl-utils jiwer pillow

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')


## 2. Acceder al dataset en Google Drive

Tu dataset debe estar en la carpeta OMR_Dataset/ de tu unidad.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATASET_PATH = '/content/drive/MyDrive/OMR_Dataset_2/dataset_entrenamiento.jsonl'

import os
print(f'Verificando dataset en: {DATASET_PATH}')
if os.path.exists(DATASET_PATH):
    print('Dataset JSONL encontrado correctamente.')
else:
    raise FileNotFoundError(
        f'No se encontró el dataset en {DATASET_PATH}.\n'
        'Revisa que la carpeta OMR_Dataset_2 exista en tu Google Drive.'
    )


## 3. Instalar scripts del pipeline OMR
Las siguientes celdas escriben los scripts Python necesarios en el entorno Colab.

In [ ]:
%%writefile /content/omr_metrics.py
import re
from jiwer import wer as _wer, cer as _cer

_NON_SYMBOL_PREFIXES = (
    "stem:", "beam:", "voice:", "staff:",
)
_NON_SYMBOL_EXACT = {
    "beam:begin", "beam:end", "beam:continue", "beam:backward-hook",
    "beam:forward-hook", "stem:up", "stem:down", "stem:none",
}


def normalize_lmx(text: str) -> str:
    if not text:
        return ""
    for marker in ["```lmx", "```xml", "```json", "```"]:
        if marker in text:
            parts = text.split(marker)
            if len(parts) >= 2:
                text = parts[1].split("```")[0]
            break
    lines = [
        l.strip() for l in text.strip().split("\n")
        if l.strip()
        and not l.strip().startswith("<?xml")
        and not l.strip().startswith("<!DOCTYPE")
    ]
    return re.sub(r"\s+", " ", " ".join(lines)).strip()


def _symbol_tokens(text: str) -> str:
    out = []
    for tok in text.split():
        if tok in _NON_SYMBOL_EXACT:
            continue
        if tok.startswith(_NON_SYMBOL_PREFIXES):
            continue
        out.append(tok)
    return " ".join(out)


def _edit_rate(ref_tokens, hyp_tokens) -> float:
    r, h = ref_tokens, hyp_tokens
    if len(r) == 0:
        return 0.0 if len(h) == 0 else 1.0
    # DP Levenshtein
    prev = list(range(len(h) + 1))
    for i, rt in enumerate(r, 1):
        cur = [i] + [0] * len(h)
        for j, ht in enumerate(h, 1):
            cost = 0 if rt == ht else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[-1] / len(r)


def compute_metrics(gt_raw: str, hyp_raw: str) -> dict:
    gt = normalize_lmx(gt_raw)
    hyp = normalize_lmx(hyp_raw)
    if not gt:
        return {"wer": None, "cer": None, "ser": None, "accuracy": None,
                "gt": gt, "hyp": hyp}

    w = float(_wer(gt, hyp))
    c = float(_cer(gt, hyp))
    s = _edit_rate(_symbol_tokens(gt).split(), _symbol_tokens(hyp).split())

    accuracy = max(0.0, min(1.0, 1.0 - s))

    return {
        "wer": round(w, 4),
        "cer": round(c, 4),
        "ser": round(s, 4),
        "accuracy": round(accuracy, 4),
        "gt": gt,
        "hyp": hyp,
    }


def classify_accuracy(accuracy: float) -> dict:
    if accuracy is None:
        return {"label": "unknown", "reliable": False,
                "message": "No se pudo evaluar la transcripcion."}
    pct = round(accuracy * 100, 1)
    if accuracy >= 0.90:
        return {"label": "excelente", "reliable": True,
                "message": f"Transcripcion fiable ({pct}% de acierto)."}
    if accuracy >= 0.75:
        return {"label": "buena", "reliable": True,
                "message": f"Transcripcion probablemente correcta ({pct}%). Revisa detalles."}
    if accuracy >= 0.50:
        return {"label": "moderada", "reliable": False,
                "message": f"Transcripcion dudosa ({pct}%). Recomendamos revision manual."}
    return {"label": "baja", "reliable": False,
            "message": f"Transcripcion poco fiable ({pct}%). Requiere correccion."}


def aggregate(records: list) -> dict:
    import statistics as st
    agg = {}
    for key in ("wer", "cer", "ser", "accuracy"):
        vals = [r[key] for r in records if r.get(key) is not None]
        if vals:
            agg[f"{key}_mean"] = round(sum(vals) / len(vals), 4)
            agg[f"{key}_std"] = round(st.pstdev(vals), 4) if len(vals) > 1 else 0.0
        else:
            agg[f"{key}_mean"] = float("nan")
            agg[f"{key}_std"] = float("nan")
    agg["n"] = len([r for r in records if r.get("ser") is not None])
    return agg


In [ ]:
%%writefile /content/fine_tune_omr.py
import os
import argparse
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

SYSTEM_PROMPT_TRAINING = "Transcribe this musical score to LMX format."


def make_normalizer(base_dir: str):
    def normalize(example):
        msgs = example.get("messages", [])
        images = example.get("images", []) or []

        img_paths = []
        for k in images:
            p = k if os.path.isabs(k) else os.path.join(base_dir, k)
            if os.path.exists(p):
                img_paths.append(p)

        user_text = SYSTEM_PROMPT_TRAINING
        assistant_text = ""
        for m in msgs:
            role = m.get("role")
            content = m.get("content", "")
            if isinstance(content, list):
                # ya estructurado: extraer el texto
                content = " ".join(
                    part.get("text", "") for part in content
                    if isinstance(part, dict) and part.get("type") == "text"
                )
            content = str(content).replace("<image>", "").strip()
            if role == "user" and content:
                user_text = content
            elif role == "assistant":
                assistant_text = content

        user_content = [{"type": "image", "image": p} for p in img_paths]
        user_content.append({"type": "text", "text": user_text})

        return {
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": [{"type": "text", "text": assistant_text}]},
            ]
        }
    return normalize

def get_best_gpu() -> int:
    import torch, subprocess
    if not torch.cuda.is_available():
        return 0
    best_vram, best_id = -1, 0
    for i in range(torch.cuda.device_count()):
        try:
            res = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.free",
                 "--format=csv,noheader,nounits", "-i", str(i)])
            free = int(res.decode().strip())
            logger.info(f"  GPU {i}: {free / 1024:.1f} GB libres")
            if free > best_vram:
                best_vram, best_id = free, i
        except Exception:
            continue
    return best_id

def run_finetuning(
    jsonl_path: str,
    model_name: str = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    output_dir: str = "qwen2.5_vl_omr_lora_v2",
    max_steps: int = -1,
    num_epochs: int = 3,
    lora_rank: int = 16,
    from_checkpoint: str = None,
    auto_steps: bool = False,
    batch_size: int = 2,
    grad_accum: int = 4,
    max_image_size: int = 1024,
):
    import torch
    from unsloth import FastVisionModel
    from unsloth.trainer import UnslothVisionDataCollator
    from datasets import load_dataset
    from trl import SFTTrainer, SFTConfig
    best_gpu = get_best_gpu()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best_gpu)
    logger.info(f"Usando GPU {best_gpu}")
    model_to_load = from_checkpoint if from_checkpoint else model_name
    logger.info(f"Cargando modelo: {model_to_load}")
    model, tokenizer = FastVisionModel.from_pretrained(
        model_to_load,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )

    if from_checkpoint:
        logger.info(f"Continuando desde checkpoint LoRA: {from_checkpoint}")
    else:
        model = FastVisionModel.get_peft_model(
            model,
            finetune_vision_layers=True,    
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=lora_rank,
            lora_alpha=lora_rank,
            lora_dropout=0,
            bias="none",
            random_state=3407,
            use_rslora=False,
        )
        logger.info("Adaptadores LoRA anadidos al modelo base.")

    logger.info(f"Cargando dataset: {jsonl_path}")
    raw_dataset = load_dataset("json", data_files=jsonl_path, split="train")
    n = len(raw_dataset)
    logger.info(f"  {n} ejemplos cargados.")
    base_dir = os.path.dirname(os.path.abspath(jsonl_path))
    logger.info("Normalizando dataset al formato de conversacion (rapido, sin abrir imagenes)...")
    converted_dataset = raw_dataset.map(
        make_normalizer(base_dir),
        batched=False,
        num_proc=4,
        remove_columns=raw_dataset.column_names,   
        desc="Normalizando",
    )

    eff_batch = batch_size * grad_accum
    steps_per_epoch = max(1, n // eff_batch)
    if auto_steps:
        max_steps = -1
        num_epochs = num_epochs if num_epochs else 3
        logger.info(f"  auto-steps -> {num_epochs} epocas "
                    f"(~{steps_per_epoch} pasos/epoca, ~{steps_per_epoch*num_epochs} pasos totales)")
    elif max_steps and max_steps > 0:
        num_epochs = -1  # SFTConfig usa max_steps
        logger.info(f"  Pasos fijados a: {max_steps} "
                    f"(~{max_steps/steps_per_epoch:.2f} epocas)")
    else:
        logger.info(f"  Entrenando {num_epochs} epocas "
                    f"(~{steps_per_epoch*num_epochs} pasos totales)")

    FastVisionModel.for_training(model)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(
            model, tokenizer,
            resize=max_image_size,   
        ),
        train_dataset=converted_dataset,
        args=SFTConfig(
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=grad_accum,
            warmup_steps=max(10, steps_per_epoch // 20),
            max_steps=max_steps if (max_steps and max_steps > 0) else -1,
            num_train_epochs=num_epochs if num_epochs and num_epochs > 0 else 1,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=5,
            save_strategy="steps",
            save_steps=50,
            save_total_limit=3,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=3407,
            output_dir=output_dir,
            report_to="none",
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_seq_length=2048,
        ),
    )

    logger.info("Iniciando entrenamiento...")
    trainer_stats = trainer.train()

    logger.info(f"Guardando modelo en {output_dir}...")
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    tl = getattr(trainer_stats, "training_loss", "N/A")
    logger.info(f"Fine-tuning completado. Training loss final: {tl}")
    logger.info(f"Modelo guardado en: {output_dir}")
    return trainer_stats

if __name__ == "__main__":
    p = argparse.ArgumentParser(description="Fine-tuning OMR con Unsloth (Qwen2.5-VL)")
    p.add_argument("--dataset", "-d", required=True, help="Ruta al JSONL de entrenamiento")
    p.add_argument("--model", "-m", default="unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit")
    p.add_argument("--output", "-o", default="qwen2.5_vl_omr_lora_v2")
    p.add_argument("--steps", "-s", type=int, default=-1,
                   help="max_steps explicitos. -1 = usar epocas (default)")
    p.add_argument("--epochs", "-e", type=int, default=3, help="Numero de epocas (default: 3)")
    p.add_argument("--rank", "-r", type=int, default=16)
    p.add_argument("--from-checkpoint", "-c", default=None)
    p.add_argument("--auto-steps", action="store_true",
                   help="Entrena por epocas automaticamente (equivale a --epochs 3)")
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--grad-accum", type=int, default=4)
    args = p.parse_args()

    if not os.path.exists(args.dataset):
        logger.error(f"Dataset no encontrado: {args.dataset}")
        raise SystemExit(1)

    logger.info("=" * 60)
    logger.info("  Fine-tuning OMR Sistema-por-Sistema")
    logger.info("=" * 60)
    logger.info(f"  Dataset:    {args.dataset}")
    logger.info(f"  Modelo:     {args.model}")
    logger.info(f"  Checkpoint: {args.from_checkpoint or 'ninguno'}")
    logger.info(f"  Salida:     {args.output}")
    logger.info(f"  LoRA r:     {args.rank}")
    logger.info("=" * 60)

    run_finetuning(
        jsonl_path=args.dataset,
        model_name=args.model,
        output_dir=args.output,
        max_steps=args.steps,
        num_epochs=args.epochs,
        lora_rank=args.rank,
        from_checkpoint=args.from_checkpoint,
        auto_steps=args.auto_steps,
        batch_size=args.batch_size,
        grad_accum=args.grad_accum,
    )


In [ ]:
%%writefile /content/evaluate_omr.py
import os, json, csv, argparse, logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PROMPT = "Transcribe this musical score to LMX format."


def get_gt(sample):
    for m in sample.get("messages", []):
        if m.get("role") == "assistant":
            c = m.get("content", "")
            if isinstance(c, list):
                return " ".join(p.get("text", "") for p in c
                                if isinstance(p, dict) and p.get("type") == "text")
            return c
    if "conversations" in sample and len(sample["conversations"]) >= 2:
        return sample["conversations"][1].get("value", "")
    return ""


def get_img_path(sample, base_dir):
    ruta = ""
    if sample.get("images"):
        ruta = sample["images"][0]
    else:
        ruta = sample.get("image", "")
    if ruta and not os.path.isabs(ruta):
        ruta = os.path.join(base_dir, ruta)
    return ruta


def run_eval(model_path, dataset_path, output_dir, n_samples=None):
    import torch
    from PIL import Image
    from unsloth import FastVisionModel
    from qwen_vl_utils import process_vision_info
    from omr_metrics import compute_metrics, classify_accuracy, aggregate

    base_dir = os.path.dirname(os.path.abspath(dataset_path))
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    logger.info(f"Cargando modelo desde {model_path} ...")
    model, tok = FastVisionModel.from_pretrained(model_path, load_in_4bit=True)
    FastVisionModel.for_inference(model)
    logger.info("Modelo listo.")

    with open(dataset_path) as f:
        samples = [json.loads(l) for l in f if l.strip()]
    if n_samples:
        samples = samples[:n_samples]
    logger.info(f"Evaluando {len(samples)} muestras...")

    records, rows = [], []
    for idx, sample in enumerate(samples):
        gt_raw = get_gt(sample)
        img_path = get_img_path(sample, base_dir)
        source = sample.get("source_file", os.path.basename(img_path) or f"sample_{idx}")

        if not gt_raw:
            continue
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            logger.warning(f"[{idx}] imagen no encontrada: {e}")
            rows.append({"idx": idx, "source": source, "wer": None, "cer": None,
                         "ser": None, "accuracy": None, "label": "error",
                         "error": str(e), "gt_preview": "", "hyp_preview": ""})
            continue

        msgs = [{"role": "user", "content": [
            {"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
        text_in = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        img_in, _ = process_vision_info(msgs)
        inputs = tok(text=[text_in], images=img_in, padding=True,
                     return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=768, do_sample=False)  # greedy
        hyp_raw = tok.batch_decode(out[:, inputs.input_ids.shape[1]:],
                                   skip_special_tokens=True)[0]

        m = compute_metrics(gt_raw, hyp_raw)
        cls = classify_accuracy(m["accuracy"])
        records.append(m)
        rows.append({"idx": idx, "source": source,
                     "wer": m["wer"], "cer": m["cer"], "ser": m["ser"],
                     "accuracy": m["accuracy"], "label": cls["label"], "error": None,
                     "gt_preview": m["gt"][:80], "hyp_preview": m["hyp"][:80]})
        logger.info(f"[{idx+1}/{len(samples)}] {source}  "
                    f"WER={m['wer']} CER={m['cer']} SER={m['ser']} acc={m['accuracy']} ({cls['label']})")

    # CSV
    csv_path = os.path.join(output_dir, "omr_evaluation.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["idx", "source", "wer", "cer", "ser",
                                          "accuracy", "label", "error",
                                          "gt_preview", "hyp_preview"])
        w.writeheader(); w.writerows(rows)
    logger.info(f"CSV guardado: {csv_path}")

    # Reporte
    agg = aggregate(records)
    exact = sum(1 for r in records if r["ser"] == 0.0)
    reliable = sum(1 for r in records if r["accuracy"] is not None and r["accuracy"] >= 0.75)
    report = f"""# Reporte de Evaluacion OMR

**Modelo:** `{model_path}`
**Dataset:** `{dataset_path}`
**Fecha:** {datetime.now().strftime('%Y-%m-%d %H:%M')}
**Muestras con metrica:** {agg['n']}

| Metrica  | Media  | Std    |
|----------|--------|--------|
| WER      | {agg['wer_mean']} | {agg['wer_std']} |
| CER      | {agg['cer_mean']} | {agg['cer_std']} |
| SER      | {agg['ser_mean']} | {agg['ser_std']} |
| Accuracy | {agg['accuracy_mean']} | {agg['accuracy_std']} |

- **Transcripciones exactas (SER=0):** {exact}/{agg['n']}
- **Fiables (accuracy >= 0.75):** {reliable}/{agg['n']}

## Interpretacion
- SER es la metrica principal para OMR (tasa de error de simbolo musical).
- accuracy = 1 - SER, se muestra al usuario como % de acierto.
- Umbrales de notificacion: >=90% excelente | >=75% buena | >=50% moderada | <50% baja.
"""
    md_path = os.path.join(output_dir, "omr_evaluation.md")
    with open(md_path, "w") as f:
        f.write(report)
    print(report)
    return agg


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--dataset", required=True)
    p.add_argument("--output-dir", default="/content/results")
    p.add_argument("--n-samples", type=int, default=None)
    a = p.parse_args()
    run_eval(a.model, a.dataset, a.output_dir, a.n_samples)


In [ ]:
%%writefile /content/cross_validate.py

import os, json, random, re, argparse, logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)


def _group_key(line):
    rec = json.loads(line)
    img = rec.get("images") or rec.get("image") or ""
    if isinstance(img, list):
        img = img[0] if img else ""
    m = re.match(r"^(.*)_v\d+_", img)
    return m.group(1) if m else img


def split_jsonl(jsonl_path, n_folds=5, seed=42):
    with open(jsonl_path, encoding="utf-8") as f:
        lines = [l for l in f if l.strip()]

    groups = {}
    for l in lines:
        groups.setdefault(_group_key(l), []).append(l)
    group_keys = list(groups.keys())

    random.seed(seed)
    random.shuffle(group_keys)
    fold_size = len(group_keys) // n_folds
    folds = []
    for i in range(n_folds):
        a = i * fold_size
        b = (i + 1) * fold_size if i < n_folds - 1 else len(group_keys)
        test_keys = set(group_keys[a:b])
        test = [l for k in group_keys[a:b] for l in groups[k]]
        train = [l for k in group_keys if k not in test_keys for l in groups[k]]
        folds.append((train, test))

    logger.info(f"{len(lines)} entradas en {len(group_keys)} grupos base -> {n_folds} folds "
                f"(agrupado por sistema para evitar fuga de variantes de augmentacion; "
                f"~{fold_size} grupos test / ~{len(group_keys)-fold_size} grupos train por fold)")
    return folds


def run_cv(dataset, output_dir, n_folds=5, epochs=2, lora_rank=16,
           model_name="unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit", eval_samples=None):
    import gc, torch
    from fine_tune_omr import run_finetuning
    from evaluate_omr import run_eval

    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    folds = split_jsonl(dataset, n_folds)
    fold_aggs = []

    for i, (train_lines, test_lines) in enumerate(folds):
        fdir = out / f"fold_{i}"; fdir.mkdir(parents=True, exist_ok=True)
        train_path = fdir / f"train_fold_{i}.jsonl"
        test_path = fdir / f"test_fold_{i}.jsonl"
        train_path.write_text("".join(train_lines), encoding="utf-8")
        test_path.write_text("".join(test_lines), encoding="utf-8")
        model_out = fdir / "model"

        logger.info(f"\n{'='*55}\nFOLD {i}  entrenamiento ({len(train_lines)} train)\n{'='*55}")
        run_finetuning(
            jsonl_path=str(train_path),
            model_name=model_name,
            output_dir=str(model_out),
            num_epochs=epochs,
            max_steps=-1,
            lora_rank=lora_rank,
        )

        logger.info(f"FOLD {i}  evaluacion ({len(test_lines)} test)")
        agg = run_eval(str(model_out), str(test_path), str(fdir / "eval"),
                       n_samples=eval_samples)
        agg["fold"] = i
        fold_aggs.append(agg)

        # liberar VRAM entre folds
        gc.collect(); torch.cuda.empty_cache()

    def mean_std(key):
        vals = [a[key] for a in fold_aggs if a.get(key) == a.get(key)]  # descarta nan
        if not vals:
            return float("nan"), float("nan")
        import statistics as st
        return (round(sum(vals)/len(vals), 4),
                round(st.pstdev(vals), 4) if len(vals) > 1 else 0.0)

    lines = ["# Reporte 5-Fold Cross Validation OMR\n",
             f"**Dataset:** `{dataset}`  ",
             f"**Fecha:** {datetime.now().strftime('%Y-%m-%d %H:%M')}  ",
             f"**Folds:** {n_folds} | **Epocas/fold:** {epochs}\n",
             "## Resultados por fold\n",
             "| Fold | WER | CER | SER | Accuracy | n |",
             "|------|-----|-----|-----|----------|---|"]
    for a in fold_aggs:
        lines.append(f"| {a['fold']} | {a['wer_mean']} | {a['cer_mean']} | "
                     f"{a['ser_mean']} | {a['accuracy_mean']} | {a['n']} |")

    lines.append("\n## Resumen global (media ± std entre folds)\n")
    lines.append("| Metrica | Media | Std |")
    lines.append("|---------|-------|-----|")
    for key, label in [("wer_mean", "WER"), ("cer_mean", "CER"),
                       ("ser_mean", "SER"), ("accuracy_mean", "Accuracy")]:
        m, s = mean_std(key)
        lines.append(f"| {label} | {m} | {s} |")

    report = "\n".join(lines) + "\n"
    (out / "cross_validation_report.md").write_text(report)
    print(report)
    logger.info(f"Reporte guardado en {out/'cross_validation_report.md'}")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", required=True)
    p.add_argument("--output-dir", default="/content/cv_results")
    p.add_argument("--folds", type=int, default=5)
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--model", default="unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit")
    p.add_argument("--eval-samples", type=int, default=None)
    a = p.parse_args()
    run_cv(a.dataset, a.output_dir, a.folds, a.epochs, a.rank, a.model, a.eval_samples)


## 4. Fine-Tuning


In [ ]:

!cd /content && python fine_tune_omr.py \
    --dataset "/content/drive/MyDrive/OMR_Dataset_2/dataset_entrenamiento.jsonl" \
    --model unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit \
    --output "/content/drive/MyDrive/OMR_Dataset_2/qwen2.5_vl_omr_lora_v2" \
    --epochs 3


## 5. Evaluar el modelo 


In [ ]:
import json, random, re
from pathlib import Path

SRC = "/content/drive/MyDrive/OMR_Dataset_2/dataset_entrenamiento.jsonl"
TRAIN_OUT = "/content/drive/MyDrive/OMR_Dataset_2/dataset_train_split.jsonl"
VAL_OUT = "/content/drive/MyDrive/OMR_Dataset_2/dataset_val_split.jsonl"
VAL_FRACTION = 0.1
SEED = 42

def _group_key(line):
    rec = json.loads(line)
    img = rec.get("images") or rec.get("image") or ""
    if isinstance(img, list):
        img = img[0] if img else ""
    m = re.match(r"^(.*)_v\d+_", img)
    return m.group(1) if m else img

with open(SRC, encoding="utf-8") as f:
    lines = [l for l in f if l.strip()]

groups = {}
for l in lines:
    groups.setdefault(_group_key(l), []).append(l)

keys = list(groups.keys())
random.seed(SEED)
random.shuffle(keys)
n_val_groups = max(1, int(len(keys) * VAL_FRACTION))
val_keys = set(keys[:n_val_groups])

train_lines = [l for k in keys if k not in val_keys for l in groups[k]]
val_lines = [l for k in val_keys for l in groups[k]]

Path(TRAIN_OUT).write_text("".join(train_lines), encoding="utf-8")
Path(VAL_OUT).write_text("".join(val_lines), encoding="utf-8")

print(f"{len(groups)} sistemas base -> {len(keys) - n_val_groups} train / {n_val_groups} val (grupos)")
print(f"Filas: {len(train_lines)} train / {len(val_lines)} val (de {len(lines)} totales)")
print(f"Escrito:\n  {TRAIN_OUT}\n  {VAL_OUT}")


In [ ]:

!cd /content && python evaluate_omr.py \
    --model "/content/drive/MyDrive/OMR_Dataset_2/qwen2.5_vl_omr_lora_v2" \
    --dataset "/content/drive/MyDrive/OMR_Dataset_2/dataset_val_split.jsonl" \
    --output-dir /content/results \
    --n-samples 100


In [ ]:
import torch
import gc

try:
    del trainer
except: pass
try:
    del model
except: pass
try:
    del tokenizer
except: pass

gc.collect()

torch.cuda.empty_cache()

vram_usada = torch.cuda.memory_allocated(0) / 1024**3
print(f"Limpieza completada. Memoria VRAM en uso ahora: {vram_usada:.2f} GB (Debería ser casi 0)")


## 6. Cross-Validation 

Tarda mucho, 5× el tiempo de un fine-tune

In [ ]:


# !cd /content && python cross_validate.py \
#     --dataset "/content/drive/MyDrive/OMR_Dataset_2/dataset_entrenamiento.jsonl" \
#     --model unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit \
#     --output-dir /content/cv_results \
#     --folds 5 --epochs 2 --eval-samples 60

# Reporte final: /content/cv_results/cross_validation_report.md


## 7. Prueba de inferencia

Prueba el modelo entrenado con una imagen de pentagrama.

In [ ]:
from unsloth import FastVisionModel
from PIL import Image
from qwen_vl_utils import process_vision_info
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    '/content/drive/MyDrive/OMR_Dataset_2/qwen2.5_vl_omr_lora_v2',
    load_in_4bit=True,
    device_map={'': 0},
)
FastVisionModel.for_inference(model)

test_img = Image.open('/content/drive/MyDrive/OMR_Dataset_2/ES-1913-CT-JSV-001_01.png').convert('RGB')  # ← Ajusta si el nombre es diferente

messages = [{'role': 'user', 'content': [
    {'type': 'image', 'image': test_img},
    {'type': 'text', 'text': 'Transcribe this musical score to LMX format.'},
]}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, _ = process_vision_info(messages)
inputs = tokenizer(text=[text], images=image_inputs, padding=True, return_tensors='pt').to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False)

result = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
print('\nTranscripción LMX del modelo:')
print(result)

---
## Descargar modelo e informes



In [ ]:

import os, shutil
from google.colab import files

LORA_DIR    = '/content/drive/MyDrive/OMR_Dataset_2/qwen2.5_vl_omr_lora_v2'
RESULTS_DIR = '/content/results'

if os.path.exists(RESULTS_DIR):
    shutil.make_archive('/content/resultados_omr', 'zip', RESULTS_DIR)
    files.download('/content/resultados_omr.zip')
    print('Informes descargados: resultados_omr.zip')
else:
    print('No se encontró /content/results, ejecuta primero la celda de evaluación.')
if os.path.exists(LORA_DIR):
    print('Comprimiendo modelo LoRA (~200 MB, puede tardar 1-2 min)...')
    shutil.make_archive('/content/omr_lora_model', 'zip', LORA_DIR)
    files.download('/content/omr_lora_model.zip')
    print('Modelo LoRA descargado: omr_lora_model.zip')
else:
    print('No se encontró el modelo, ejecuta primero la celda de entrenamiento.')


